# ocrkit: read scanned documents on a Colab GPU, unattended

Reads every page in a manifest (made on your computer with `ocrkit manifest`) with current document-OCR models via vLLM. Optionally runs a bake-off first and picks the model whose tables add up best. Writes each page's output to Google Drive the moment it is read.

1. **Runtime → Change runtime type → A100** (an L4 works, about 2–3× slower; a T4 works too).
2. Fill in cell 1, then run cells 1–4. Cell 2 asks once for Drive access. Cell 3 starts the job **detached**: no button in this notebook can stop it by accident. Cell 4 shows progress and **releases the GPU when the last page is read**.
3. Afterwards, download the output folder from drive.google.com (right-click → Download; Drive zips it on its side, no GPU time) and run `ocrkit fetch` on your computer.

If the session drops, run the cells again: pages already on Drive are skipped.

## 1. Settings

In [ ]:
PROJECT  = 'myproject'                                   # a folder of this name is made in Drive/ocrkit/
MANIFEST = 'https://github.com/OWNER/REPO/releases/download/TAG/manifest.json'   # or a path in Drive
BAKEOFF  = {'models': ['ATH-MaaS/OvisOCR2', 'PaddlePaddle/PaddleOCR-VL-1.6'], 'edges': [1600], 'top_edges': [2048],
            'scorer': 'ocrkit.score:consistency'}          # None to skip; scorer: 'module:function'
RUNS     = 'winner'                                      # or [{'model': 'ATH-MaaS/OvisOCR2', 'edge': 2048}, ...]
FILES    = None                                          # or a list of file ids: only these
HOURS    = 9                                             # everything stops after this

## 2. Drive, ocrkit and vLLM (3–5 minutes)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q -U --force-reinstall --no-deps "ocrkit @ git+https://github.com/reclaimchennai/NCRB#subdirectory=ocrkit" 2>&1 | tail -1
!pip install -q -U vllm "ocrkit @ git+https://github.com/reclaimchennai/NCRB#subdirectory=ocrkit" 2>&1 | tail -2
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader
import json, subprocess
cap = float(subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'], capture_output=True, text=True).stdout.split()[0])
import os
OUT = f'/content/drive/MyDrive/ocrkit/{PROJECT}'
os.makedirs(OUT, exist_ok=True)          # the job's log goes next to it
plan = {'name': PROJECT, 'manifest': MANIFEST, 'out': OUT, 'runs': RUNS, 'hours': HOURS, 'dtype': 'auto' if cap >= 8 else 'half'}
if BAKEOFF: plan['bakeoff'] = BAKEOFF
if FILES: plan['files'] = FILES
open('/content/plan.json', 'w').write(json.dumps(plan, indent=1))
print(json.dumps(plan, indent=1))

## 3. Start the job (detached)

In [ ]:
import subprocess, time
pids = lambda: subprocess.run(['pgrep', '-f', '[o]crkit.job'], capture_output=True, text=True).stdout.split()
if pids(): print('already running', pids())
else:
    subprocess.Popen(['python', '-m', 'ocrkit.job', '/content/plan.json'], stdout=open(f'/content/drive/MyDrive/ocrkit/{PROJECT}.log', 'a'),
                     stderr=subprocess.STDOUT, start_new_session=True)
    time.sleep(20); print('started', pids())

## 4. Progress (safe to stop and re-run; releases the GPU at the end)

In [ ]:
from ocrkit.monitor import watch
watch(OUT)

## If something looks wrong

Paste the output of this cell (it only reads):

In [ ]:
print(open(f'{OUT}/status.json').read()[-2500:])
!pgrep -af "[o]crkit" | cut -c1-150; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader
!tail -15 /content/drive/MyDrive/ocrkit/{PROJECT}.log